In [1]:
import pandas as pd

In [3]:
presence_df = pd.read_csv('union_taxa_presence_table.csv')

In [9]:
contaminants_copd_yan = pd.read_csv('COPD_Yan/contaminants_species.csv')

In [17]:
overlap = presence_df[presence_df["taxon"].isin(contaminants_copd_yan["species"])]
overlap.to_csv('present_inCOPD_yan_contaminants.csv')

In [18]:
contaminants_copd_bal = pd.read_csv('Imran/contaminants.csv')

In [22]:
overlap = presence_df[presence_df["taxon"].isin(contaminants_copd_bal["taxa"])]
overlap.to_csv('present_inCOPD_bal_contaminants.csv')

In [24]:
contaminants_lungcancer_bal = pd.read_csv('LungCancer/Jan8_76patients/potential_contaminants.csv')

In [27]:
overlap = presence_df[presence_df["taxon"].isin(contaminants_lungcancer_bal["taxa"])]
overlap.to_csv('present_inLungcancer_bal_contaminants.csv')

In [28]:
overlap

,taxon,Lung cancer MG,Lung cancer MT,COPD BAL MG,COPD BAL MT,COPD sputum MG,num_dfs_present
1,Abyssibius alkaniclasticus,False,False,False,False,True,1
14,Acidovorax sp. 1608163,False,False,False,False,True,1
17,Acidovorax sp. KKS102,False,False,False,False,True,1
26,Acinetobacter guillouiae,False,True,False,False,False,1
47,Acinetobacter sp. LUNF3,False,True,False,False,True,2
113,Agrobacterium pusense,False,False,False,False,True,1
120,Alicycliphilus denitrificans,False,False,False,True,True,2
139,Anabaena sp. YBS01,False,False,True,True,False,2
440,Corynebacterium segmentosum,False,False,True,True,False,2
485,Deinococcus radiodurans,False,False,False,True,False,1


In [12]:
df_LC_MG = pd.read_csv('LungCancer/df_LC_MG_168.csv',sep = '\t',index_col=0)
df_LC_MT = pd.read_csv('LungCancer/df_LC_MT_389.csv',sep = '\t',index_col=0)
df_COPD_BAL_MT = pd.read_csv('Imran/df_COPD_BAL_MT_546.csv',sep = '\t',index_col=0)
df_COPD_BAL_MG = pd.read_csv('Imran/df_COPD_BAL_MG_190.csv',sep = '\t',index_col=0)
df_COPD_sputum_MG = pd.read_csv('COPD_Yan/df_COPD_sputum_MG_1053.csv',sep = '\t',index_col=0)

In [13]:
# Get the union of all indices
#union_index = df1.index.union(df2.index).union(df3.index).union(df4.index).union(df5.index)

# a cleaner way:
from functools import reduce

union_index = reduce(lambda x, y: x.union(y), [df_LC_MG.index, df_LC_MT.index, 
                                               df_COPD_BAL_MT.index, df_COPD_BAL_MG.index, df_COPD_sputum_MG.index])
union_index = union_index.drop('synthetic construct')
index_df = pd.DataFrame({'index': union_index})
index_df.to_csv('union_index.csv', index=False)

In [16]:
# union_index.shape

(1552,)

In [54]:
from functools import reduce
import pandas as pd

# Assuming you already have the 5 DataFrames: df_LC_MG, df_LC_MT, df_COPD_BAL_MT, df_COPD_BAL_MG, df_COPD_sputum_MG

# Get the union of all indices
union_index = reduce(lambda x, y: x.union(y), [
    df_LC_MG.index,
    df_LC_MT.index,
    df_COPD_BAL_MT.index,
    df_COPD_BAL_MG.index,
    df_COPD_sputum_MG.index
])

# Remove 'synthetic construct' if it exists
union_index = union_index.drop('synthetic construct', errors='ignore')

# Create a new DataFrame with union index
presence_df = pd.DataFrame(index=union_index)
presence_df['Lung cancer MG'] = presence_df.index.isin(df_LC_MG.index)
presence_df['Lung cancer MT'] = presence_df.index.isin(df_LC_MT.index)
presence_df['COPD BAL MG'] = presence_df.index.isin(df_COPD_BAL_MG.index)
presence_df['COPD BAL MT'] = presence_df.index.isin(df_COPD_BAL_MT.index)
presence_df['COPD sputum MG'] = presence_df.index.isin(df_COPD_sputum_MG.index)

# Optionally reset index so 'index' is a column
presence_df = presence_df.reset_index().rename(columns={'index': 'taxon'})
# Add a column that counts how many dataframes the taxon appears in
presence_df['num_dfs_present'] = (
    presence_df[['Lung cancer MG', 'Lung cancer MT', 
                 'COPD BAL MG', 'COPD BAL MT', 
                 'COPD sputum MG']].sum(axis=1)
)

presence_df
# Save to CSV
presence_df.to_csv('union_taxa_presence_table.csv', index=False)

In [56]:
presence_df

,taxon,Lung cancer MG,Lung cancer MT,COPD BAL MG,COPD BAL MT,COPD sputum MG,num_dfs_present
0,Abiotrophia defectiva,True,False,False,True,True,3
1,Abyssibius alkaniclasticus,False,False,False,False,True,1
2,Acetobacter aceti,False,False,False,False,True,1
3,Acetobacter ghanensis,False,False,False,False,True,1
4,Acholeplasma laidlawii,False,False,False,False,True,1
...,...,...,...,...,...,...,...
1547,[Candida] glabrata,False,False,False,True,False,1
1548,[Ruminococcus] gnavus,False,True,False,True,True,3
1549,[Ruminococcus] torques,False,False,False,False,True,1
1550,cyanobacterium endosymbiont of Epithemia turgida,False,False,False,True,False,1


In [57]:
filtered_df = presence_df[presence_df['num_dfs_present'] >  2]

import re

def escape_all_latex(s):
    # Escapes most LaTeX special characters
    s = str(s)
    s = s.replace('\\', r'\\textbackslash{}')
    s = s.replace('&', r'\&')
    s = s.replace('%', r'\%')
    s = s.replace('$', r'\$')
    s = s.replace('#', r'\#')
    s = s.replace('_', r'\_')
    s = s.replace('{', r'\{')
    s = s.replace('}', r'\}')
    s = s.replace('~', r'\textasciitilde{}')
    s = s.replace('^', r'\^{}')
    return s

filtered_df['taxon'] = filtered_df['taxon'].apply(escape_all_latex)
# Add \textit{} around each taxon name in the 'index' column
filtered_df['taxon'] = filtered_df['taxon'].apply(lambda x: f'\\textit{{{x}}}')

# Save to new CSV or use as needed
filtered_df.to_csv("filtered_taxa_present_in_more_than_2.csv", index=False)

/var/folders/0g/jffzlx7x0v37frp3q4ssdnp80000gn/T/ipykernel_24082/3868712066.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['taxon'] = filtered_df['taxon'].apply(escape_all_latex)
/var/folders/0g/jffzlx7x0v37frp3q4ssdnp80000gn/T/ipykernel_24082/3868712066.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['taxon'] = filtered_df['taxon'].apply(lambda x: f'\\textit{{{x}}}')


In [58]:
filtered_df.shape

(207, 7)

In [59]:
import pandas as pd

# Load your CSV
df = pd.read_csv("filtered_taxa_present_in_more_than_2.csv")

# Optional: reduce number of rows/columns for nicer formatting in LaTeX
# df = df.head(10)

# Convert to LaTeX
latex_code = df.to_latex(index=False, longtable=True, caption="Taxon Presence Across Datasets", label="tab:taxa_presence")

# Save to .tex file
with open("taxa_presence_table.tex", "w") as f:
    f.write(latex_code)

# Print preview
print(latex_code)

\begin{longtable}{lrrrrrr}
\caption{Taxon Presence Across Datasets} \label{tab:taxa_presence} \\
\toprule
taxon & Lung cancer MG & Lung cancer MT & COPD BAL MG & COPD BAL MT & COPD sputum MG & num_dfs_present \\
\midrule
\endfirsthead
\caption[]{Taxon Presence Across Datasets} \\
\toprule
taxon & Lung cancer MG & Lung cancer MT & COPD BAL MG & COPD BAL MT & COPD sputum MG & num_dfs_present \\
\midrule
\endhead
\midrule
\multicolumn{7}{r}{Continued on next page} \\
\midrule
\endfoot
\bottomrule
\endlastfoot
\textit{Abiotrophia defectiva} & True & False & False & True & True & 3 \\
\textit{Achromobacter xylosoxidans} & False & True & False & True & True & 3 \\
\textit{Acinetobacter baumannii} & True & True & False & True & True & 4 \\
\textit{Acinetobacter haemolyticus} & True & False & True & False & True & 3 \\
\textit{Acinetobacter johnsonii} & True & True & True & True & True & 5 \\
\textit{Actinomyces oris} & True & True & True & True & True & 5 \\
\textit{Actinomyces pacaensis} & T

In [17]:
index_df = pd.DataFrame({'index': union_index})
index_df.to_csv('union_index.csv', index=False)

In [18]:
loaded_index_df = pd.read_csv('union_index.csv')
loaded_index = loaded_index_df['index'].tolist()  # or .values if you want a NumPy array

In [19]:
loaded_index_df

,index
0,Abiotrophia defectiva
1,Abyssibius alkaniclasticus
2,Acetobacter aceti
3,Acetobacter ghanensis
4,Acholeplasma laidlawii
...,...
1547,[Candida] glabrata
1548,[Ruminococcus] gnavus
1549,[Ruminococcus] torques
1550,cyanobacterium endosymbiont of Epithemia turgida
